In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scipy
from PIL import Image
from skimage.measure import regionprops

from tqdm import tqdm

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# Import denoised image and nucleus masks

runs_dir = Path(r'/Volumes/gchao/bamfaile/Analysis/TUBB2B-KI/00_iNs_Batch20241011/D10/Puro10min')

mask_cell_dir = runs_dir / 'ROIs'/'Cytoplasm'
mask_cell_path = os.path.join(mask_cell_dir,'*.tif') 
mask_cell_files = np.sort(glob.glob(mask_cell_path))

# Extract the integers before the first underscore in the filenames
loaded_image_ids = {
    os.path.basename(f).split('_')[0]  # Extract the part before the first underscore
    for f in mask_cell_files
}

mask_nucleus_dir = runs_dir / 'ROIs'/'Nucleus'
mask_nucleus_path = os.path.join(mask_nucleus_dir,'*.tif') 
mask_nucleus_files = np.sort(glob.glob(mask_nucleus_path))

# Filter the new images based on matching IDs
mask_nucleus_files = np.array([
    f for f in mask_nucleus_files
    if os.path.basename(f).split('_')[0] in loaded_image_ids
])

tracks_dir = runs_dir / 'Tracking'/'Run20250116'/'Factor7Link5'
tracks_path = os.path.join(tracks_dir,'*.csv')
tracks_files = np.sort(glob.glob(tracks_path))

print(len(mask_cell_files))
print(len(mask_nucleus_files))
print(len(tracks_files))

In [ ]:
# Read images into list

mask_cell = []
mask_nucleus = []

for file in tqdm(mask_cell_files):
    image = np.array(Image.open(file))
    mask_cell.append(image)

for file in tqdm(mask_nucleus_files):
    image = np.array(Image.open(file))
    mask_nucleus.append(image)

In [ ]:
print(mask_nucleus[-1].dtype, mask_nucleus[-1].shape, np.min(mask_nucleus[-1]), np.max(mask_nucleus[-1]), np.unique(mask_nucleus[-1]))

In [ ]:
# Read csv files into list

tracks = []

for file in tqdm(tracks_files):
    tracks_all = pd.read_csv(file)
    tracks.append(tracks_all)

for i in range(len(tracks)):
    tracks[i] = tracks[i].loc[:, ~tracks[i].columns.str.contains('^Unnamed')]

tracks[0].head()

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(mask_cell[-1])
ax[1].imshow(mask_nucleus[-1])

In [ ]:
print(np.unique(mask_cell[-1]),np.unique(mask_nucleus[-1]))

In [364]:
# If necessary, match labels of cell and nucleus

for cell, nucleus in zip(mask_cell, mask_nucleus):
    
    # Create a dictionary to store coordinates and corresponding intensity values from mask_cell
    coordinates_intensity = {}

    # Store coordinates and intensity values from mask_cell in the dictionary
    for cell_label in regionprops(cell):
        for coord in cell_label.coords:
            coordinates_intensity[tuple(coord)] = cell[coord[0], coord[1]]

    # Update nucleus_image based on the mapped coordinates and intensity values
    for nucleus_label in regionprops(nucleus):
        for coord in nucleus_label.coords:
            if tuple(coord) in coordinates_intensity:
                nucleus[coord[0], coord[1]] = coordinates_intensity[tuple(coord)]

In [ ]:
# Plot results of label matching

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(mask_cell[-1])
ax[1].imshow(mask_nucleus[-1])

In [553]:
# Compute distance transform

pixel_size = 0.134  # Pixel size in micrometers
default_distance = 150  # Default distance in micrometers for ROIs without a nucleus

for csv, nucleus in zip(tracks, mask_nucleus):
    csv['distance_nucleus'] = np.nan  # Create an empty column

    for roi in csv['roi_id'].unique():
        tracks_roi = csv[csv['roi_id'] == roi]
        
        # Check if the nucleus exists for the current ROI
        if roi not in np.unique(nucleus):
            # Assign the default distance for all coordinates in this ROI
            csv.loc[csv['roi_id'] == roi, 'distance_nucleus'] = default_distance
            continue
        
        # Compute the distance transform for the nucleus mask
        mask_distance_transform = scipy.ndimage.distance_transform_edt(~(nucleus == roi))
        
        # Round and clip coordinates
        coords_round = np.round(np.array(tracks_roi[['x', 'y']])).astype(int)
        coords_round[:, 0] = np.clip(coords_round[:, 0], 0, mask_distance_transform.shape[0] - 1)
        coords_round[:, 1] = np.clip(coords_round[:, 1], 0, mask_distance_transform.shape[1] - 1)
        
        # Calculate distances in micrometers
        distances_in_um = mask_distance_transform[coords_round[:, 1], coords_round[:, 0]] * pixel_size
        csv.loc[csv['roi_id'] == roi, 'distance_nucleus'] = distances_in_um


In [ ]:
# Save updated tracks files

for df, file in zip(tracks, tracks_files):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(file))[0]
    print(file_name)
    
    # Define the output file path
    output_file = os.path.join(tracks_dir, file_name + '.csv')
    print(output_file)
    
    # Save the manipulated image as a TIFF
    df.to_csv(output_file)